In [ ]:
import numpy as np
import pandas as pd

In [ ]:
high_samples = pd.read_csv("./MotifDiff Results/DE_Annotation_MotifDiff_Pathway_High.csv");
high_samples

In [ ]:
low_samples = pd.read_csv("./MotifDiff Results/DE_Annotation_MotifDiff_Pathway_Low.csv");
low_samples

In [ ]:
def run_infectivity_pipeline(file_path):
    df = pd.read_csv(file_path)
    df['variant'] = df['CHROM'] + ":" + df['POS'].astype(str)
    
    # TF motif analysis
    tf_cols = [col for col in df.columns if col.startswith("MA")]
    df['Top_TF_ID'] = df[tf_cols].abs().idxmax(axis=1)
    df['tf_score'] = df.apply(lambda row: row[row['Top_TF_ID']], axis=1)
    df['TF'] = df['Top_TF_ID'].str.split(" ").str[-1]
    df['tf_binding'] = df['tf_score'].apply(lambda x: 'gain' if x > 0 else 'loss')
    
    # gene expression
    df['expression'] = df['log2FoldChange'].apply(lambda x: 'up' if x > 0 else 'down')
    
    # consistency check
    def check_consistency(row):
        if (row['tf_binding'] == 'gain' and row['expression'] == 'up') or \
           (row['tf_binding'] == 'loss' and row['expression'] == 'down'):
            return 'Consistent'
        return 'Complex'
    df['reg_logic'] = df.apply(check_consistency, axis=1)
    
    # filtering
    df_clean = df[(df['padj'] < 0.05) & (df['tf_score'].abs() > 20)].copy()
    
    # variant-level summary
    variant_summary = df_clean.groupby('variant').agg({
        'TF': lambda x: list(set(x)),
        'SYMBOL': lambda x: list(set(x)),
        'cCRE_Type': lambda x: list(set(x)),
        'IMPACT': lambda x: list(set(x)),
        'expression': lambda x: list(set(x)),
        'log2FoldChange': lambda x: list(set(x)),
        'Pathway': lambda x: list(set(x)),
        'tf_score': lambda x: list(set(x)),
        'reg_logic': lambda x: list(set(x)),
        'gnomADg_AF': 'mean', 
        'REF': lambda x: list(set(x)),
        'ALT': lambda x: list(set(x)),

            # if tracking the samples carrying the variant, uncomment the code below:
        #'Carrying_Samples': lambda x: list(set(x)),
        #'All_Samples_GT': lambda x: list(set(x))
    }).sort_values(by='gnomADg_AF', ascending=True)
    
    return variant_summary

In [ ]:
high_summary = run_infectivity_pipeline("./MotifDiff Results/DE_Annotation_MotifDiff_Pathway_High.csv");
low_summary = run_infectivity_pipeline("./MotifDiff Results/DE_Annotation_MotifDiff_Pathway_Low.csv");

In [ ]:
high_summary.to_csv("./PrioritizedCandidates_High.csv, index = TRUE)
low_summary.to_csv("./PrioritizedCandidates_Low.csv, index = TRUE)